# Job Scam Detection — Dataset Integration (3-Class, Merged)

This notebook merges the two prior integration notebooks into a single
pipeline. It loads all six original job-posting datasets plus the 3-class
synthetic dataset directly, standardizes their schemas, harmonizes the
target into three classes, and exports one modelling-ready file.

**No intermediate zip file is used.** The previous version of this pipeline
split into two notebooks connected by a zipped CSV export
(`final_job_spam_dataset.csv.zip`); that step is removed here — everything
runs start to finish in one notebook.

**Target scheme — `is_fraud` has three classes:**

- **0 — Legitimate**
- **1 — Suspicious**
- **2 — Fraudulent / Scam**

**Important, carried over from the original integration notebook:** real
job-board postings (Fuzu, PigiaMe, Corporate Staffing, BrighterMonday,
JobWeb) are **not** automatically labeled legitimate — their `is_fraud`
stays `NaN` unless the project team explicitly approves that labeling
assumption (see the optional, commented-out cell near the end). Only the
Fake Job Postings (EMSCAD) dataset has genuine ground-truth labels among the
real data; those are recovered from its original `fraudulent` column, not
guessed.

**Synthetic data note:** the synthetic file already uses the three-class
scheme (0/1/2) natively — no relabeling is needed for it, only for the
binary EMSCAD labels. Synthetic performance should not be treated as
evidence of real-world model performance; a held-out real, labeled sample
should be used for final evaluation where possible.


In [1]:
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)


## 1. Load the datasets

All seven source files are loaded directly — no zip step.

In [2]:
# Six original datasets
df_fake = pd.read_csv('fake_job_postings.csv')
df_fuzu = pd.read_csv('fuzu_kenya_jobs.csv')
df_pigia = pd.read_csv('pigiame_jobs_full.csv')
df_corp = pd.read_csv('corporatestaffing_jobs_full.csv')
df_bm = pd.read_csv('brightermonday_kenya_jobs.csv')
df_jobweb = pd.read_csv('jobwebkenya_jobs_full.csv')

# Synthetic dataset -- already uses the 3-class is_fraud scheme (0/1/2)
df_synthetic = pd.read_csv('synthetic_40k.csv')

datasets = {
    'Fake Job Postings': df_fake,
    'Fuzu Kenya': df_fuzu,
    'PigiaMe Kenya': df_pigia,
    'Corporate Staffing Kenya': df_corp,
    'BrighterMonday Kenya': df_bm,
    'JobWeb Kenya': df_jobweb,
    'Synthetic Augmentation': df_synthetic,
}

for name, df in datasets.items():
    print(f'{name}: {df.shape}')


Fake Job Postings: (17880, 18)
Fuzu Kenya: (666, 7)
PigiaMe Kenya: (1500, 8)
Corporate Staffing Kenya: (3988, 7)
BrighterMonday Kenya: (1964, 13)
JobWeb Kenya: (5323, 9)
Synthetic Augmentation: (40000, 13)


## 2. Clean column names

In [3]:
for df in datasets.values():
    df.columns = df.columns.str.strip()

for name, df in datasets.items():
    print(f'\n{name}')
    print(df.columns.tolist())



Fake Job Postings
['job_id', 'title', 'location', 'department', 'salary_range', 'company_profile', 'description', 'requirements', 'benefits', 'telecommuting', 'has_company_logo', 'has_questions', 'employment_type', 'required_experience', 'required_education', 'industry', 'function', 'fraudulent']

Fuzu Kenya
['title', 'company', 'location', 'salary_range', 'tags', 'description', 'source_url']

PigiaMe Kenya
['title', 'company', 'work_type', 'location', 'salary_range', 'posted', 'description', 'source_url']

Corporate Staffing Kenya
['title', 'industry', 'salary_range', 'location', 'deadline', 'description', 'source_url']

BrighterMonday Kenya
['title', 'company', 'category', 'location', 'work_type', 'salary_range', 'min_qualification', 'experience_level', 'experience_length', 'language_requirement', 'working_hours', 'description', 'source_url']

JobWeb Kenya
['title', 'company', 'location', 'state', 'job_type', 'job_category', 'closing_date', 'description', 'source_url']

Synthetic Au

## 3. Standardize each dataset into a common schema

Each source is mapped into the same set of columns. Fields the source
doesn't have are filled with `NaN` rather than guessed.


In [4]:
# 1. Fake Job Postings (EMSCAD) -- has genuine ground-truth labels
df_fake_std = pd.DataFrame({
    'title': df_fake['title'],
    'company': df_fake['company_profile'],
    'location': df_fake['location'],
    'industry_category': df_fake['industry'],
    'work_type': df_fake['employment_type'],
    'salary_range': df_fake['salary_range'],
    'experience_level': df_fake['required_experience'],
    'min_qualification': df_fake['required_education'],
    'description': df_fake['description'],
    'requirements': df_fake['requirements'],
    'source_url': np.nan,
    'source_platform': 'Fake Job Postings Dataset',
    'is_fraud': df_fake['fraudulent'],   # real label, recovered from the original column
    'data_source': 'original',
})

# 2. Fuzu Kenya -- no fraud label available
df_fuzu_std = pd.DataFrame({
    'title': df_fuzu['title'],
    'company': df_fuzu['company'],
    'location': df_fuzu['location'],
    'industry_category': df_fuzu['tags'],
    'work_type': np.nan,
    'salary_range': df_fuzu['salary_range'],
    'experience_level': np.nan,
    'min_qualification': np.nan,
    'description': df_fuzu['description'],
    'requirements': np.nan,
    'source_url': df_fuzu['source_url'],
    'source_platform': 'Fuzu Kenya',
    'is_fraud': np.nan,
    'data_source': 'original',
})

# 3. PigiaMe Kenya -- no fraud label available
df_pigia_std = pd.DataFrame({
    'title': df_pigia['title'],
    'company': df_pigia['company'],
    'location': df_pigia['location'],
    'industry_category': np.nan,
    'work_type': df_pigia['work_type'],
    'salary_range': df_pigia['salary_range'],
    'experience_level': np.nan,
    'min_qualification': np.nan,
    'description': df_pigia['description'],
    'requirements': np.nan,
    'source_url': df_pigia['source_url'],
    'source_platform': 'PigiaMe Kenya',
    'is_fraud': np.nan,
    'data_source': 'original',
})

# 4. Corporate Staffing Kenya -- no fraud label available
df_corp_std = pd.DataFrame({
    'title': df_corp['title'],
    'company': np.nan,
    'location': df_corp['location'],
    'industry_category': df_corp['industry'],
    'work_type': np.nan,
    'salary_range': df_corp['salary_range'],
    'experience_level': np.nan,
    'min_qualification': np.nan,
    'description': df_corp['description'],
    'requirements': np.nan,
    'source_url': df_corp['source_url'],
    'source_platform': 'Corporate Staffing Kenya',
    'is_fraud': np.nan,
    'data_source': 'original',
})


In [5]:
# 5. BrighterMonday Kenya -- no fraud label available
df_bm_std = pd.DataFrame({
    'title': df_bm['title'],
    'company': df_bm['company'],
    'location': df_bm['location'],
    'industry_category': df_bm['category'],
    'work_type': df_bm['work_type'],
    'salary_range': df_bm['salary_range'],
    'experience_level': df_bm['experience_level'],
    'min_qualification': df_bm['min_qualification'],
    'description': df_bm['description'],
    'requirements': np.nan,
    'source_url': df_bm['source_url'],
    'source_platform': 'BrighterMonday Kenya',
    'is_fraud': np.nan,
    'data_source': 'original',
})


In [6]:
# 6. JobWeb Kenya -- no fraud label available
df_jobweb_std = pd.DataFrame({
    'title': df_jobweb['title'],
    'company': df_jobweb['company'],
    'location': df_jobweb['location'],
    'industry_category': df_jobweb['job_category'],
    'work_type': df_jobweb['job_type'],
    'salary_range': np.nan,
    'experience_level': np.nan,
    'min_qualification': np.nan,
    'description': df_jobweb['description'],
    'requirements': np.nan,
    'source_url': df_jobweb['source_url'],
    'source_platform': 'JobWeb Kenya',
    'is_fraud': np.nan,
    'data_source': 'original',
})


## 4. Standardize the synthetic dataset

`synthetic_40k.csv` already uses the project's standard column names
directly, and its `is_fraud` column already uses the 0/1/2 scheme — so this
is a straight pass-through with a `data_source` tag added, not a remapping.


In [7]:
df_synthetic_std = pd.DataFrame({
    'title': df_synthetic['title'],
    'company': df_synthetic['company'],
    'location': df_synthetic['location'],
    'industry_category': df_synthetic['industry_category'],
    'work_type': df_synthetic['work_type'],
    'salary_range': df_synthetic['salary_range'],
    'experience_level': df_synthetic['experience_level'],
    'min_qualification': df_synthetic['min_qualification'],
    'description': df_synthetic['description'],
    'requirements': df_synthetic['requirements'],
    'source_url': np.nan,
    'source_platform': 'Synthetic',
    'is_fraud': df_synthetic['is_fraud'],   # already 0 / 1 / 2 -- no remapping needed
    'data_source': 'synthetic',
})


## 5. Combine all datasets

A vertical concatenation — every dataset now shares the same standardized
schema, so we stack rows.


In [8]:
final_df = pd.concat(
    [df_fake_std, df_fuzu_std, df_pigia_std, df_corp_std, df_bm_std, df_jobweb_std, df_synthetic_std],
    ignore_index=True
)

print('Combined shape:', final_df.shape)
final_df.head(3)


Combined shape: (71321, 14)


,title,company,location,industry_category,work_type,salary_range,experience_level,min_qualification,description,requirements,source_url,source_platform,is_fraud,data_source
0,Marketing Intern,"We're Food52, and we've created a groundbreaking and award-winning cooking site. We support, con...","US, NY, New York",NaN,Other,NaN,Internship,NaN,"Food52, a fast-growing, James Beard Award-winning online food community and crowd-sourced and cu...",Experience with content management systems a major plus (any blogging counts!)Familiar with the ...,NaN,Fake Job Postings Dataset,0.0,original
1,Customer Service - Cloud Video Production,"90 Seconds, the worlds Cloud Video Production Service.90 Seconds is the worlds Cloud Video Produ...","NZ, , Auckland",Marketing and Advertising,Full-time,NaN,Not Applicable,NaN,Organised - Focused - Vibrant - Awesome!Do you have a passion for customer service? Slick typing...,"What we expect from you:Your key responsibility will be to communicate with the client, 90 Secon...",NaN,Fake Job Postings Dataset,0.0,original
2,Commissioning Machinery Assistant (CMA),Valor Services provides Workforce Solutions that meet the needs of companies across the Private ...,"US, IA, Wever",NaN,NaN,NaN,NaN,NaN,"Our client, located in Houston, is actively seeking an experienced Commissioning Machinery Assis...",Implement pre-commissioning and commissioning procedures for rotary equipment.Execute all activi...,NaN,Fake Job Postings Dataset,0.0,original


## 6. Harmonize `is_fraud` into three classes

- The **EMSCAD** portion uses a binary scheme where `1` means fraudulent.
  Known fraudulent records are mapped `1 → 2`; known legitimate records stay `0`.
- The **Kenya job-board** portions have no label at all and remain `NaN` —
  not silently assumed legitimate.
- The **synthetic** portion already uses 0/1/2 natively and passes through unchanged.


In [9]:
final_df['is_fraud'] = pd.to_numeric(final_df['is_fraud'], errors='coerce')

# Only the EMSCAD rows are still on the old binary scheme at this point;
# remap just that source so the Kenya-board NaNs and the synthetic 0/1/2
# values are left untouched.
is_emscad = final_df['source_platform'] == 'Fake Job Postings Dataset'
final_df.loc[is_emscad, 'is_fraud'] = final_df.loc[is_emscad, 'is_fraud'].map({0: 0, 1: 2})

class_names = {0: 'Legitimate', 1: 'Suspicious', 2: 'Fraudulent / Scam'}

print('is_fraud value counts after harmonization:')
print(final_df['is_fraud'].value_counts(dropna=False).sort_index())


is_fraud value counts after harmonization:
is_fraud
0.0    39871
1.0    11429
2.0     6580
NaN    13441
Name: count, dtype: int64


## 7. Inspect fraud labels by source

A key sanity check before modelling: confirms the Kenya job-board sources
are still unlabeled (`NaN`), EMSCAD has real 0/2 labels, and the synthetic
data carries all three classes.


In [10]:
fraud_by_source = pd.crosstab(
    final_df['source_platform'],
    final_df['is_fraud'],
    dropna=False
)
display(fraud_by_source)


is_fraud,0.0,1.0,2.0,NaN
source_platform,,,,
BrighterMonday Kenya,0,0,0,1964
Corporate Staffing Kenya,0,0,0,3988
Fake Job Postings Dataset,17014,0,866,0
Fuzu Kenya,0,0,0,666
JobWeb Kenya,0,0,0,5323
PigiaMe Kenya,0,0,0,1500
Synthetic,22857,11429,5714,0


## 8. Optional labeling decision

Left commented out deliberately. Only uncomment this if the project team
explicitly decides to treat unlabeled real job-board postings as
presumed-legitimate (class 0) — this is a labeling **policy** decision, not
a default.


In [11]:
# Example ONLY -- uncomment after the labeling policy has been approved.

# legitimate_platforms = [
#     'Fuzu Kenya',
#     'PigiaMe Kenya',
#     'Corporate Staffing Kenya',
#     'BrighterMonday Kenya',
#     'JobWeb Kenya',
# ]
# final_df.loc[
#     final_df['source_platform'].isin(legitimate_platforms),
#     'is_fraud'
# ] = 0


## 9. Remove exact duplicate postings

In [12]:
before = len(final_df)

final_df = final_df.drop_duplicates(
    subset=['title', 'company', 'location', 'description'],
    keep='first'
).reset_index(drop=True)

after = len(final_df)
print('Rows before duplicate removal:', before)
print('Rows after duplicate removal:', after)
print('Duplicates removed:', before - after)


Rows before duplicate removal: 71321
Rows after duplicate removal: 70302
Duplicates removed: 1019


## 10. Full dataset summary (before trimming metadata columns)

In [13]:
print('Shape:', final_df.shape)

print('\nSource platform distribution:')
display(final_df['source_platform'].value_counts())

print('\nData source distribution:')
display(final_df['data_source'].value_counts())

print('\nFraud label distribution:')
display(final_df['is_fraud'].value_counts(dropna=False).sort_index())

print('\nNamed classes:')
for label, name in class_names.items():
    count = (final_df['is_fraud'] == label).sum()
    print(f'{label}: {name} -> {count:,}')
print(f"NaN / unknown labels -> {final_df['is_fraud'].isna().sum():,}")


Shape: (70302, 14)

Source platform distribution:


source_platform
Synthetic                    39733
Fake Job Postings Dataset    17417
JobWeb Kenya                  5257
Corporate Staffing Kenya      3982
BrighterMonday Kenya          1954
PigiaMe Kenya                 1293
Fuzu Kenya                     666
Name: count, dtype: int64


Data source distribution:


data_source
synthetic    39733
original     30569
Name: count, dtype: int64


Fraud label distribution:


is_fraud
0.0    39219
1.0    11426
2.0     6505
NaN    13152
Name: count, dtype: int64


Named classes:
0: Legitimate -> 39,219
1: Suspicious -> 11,426
2: Fraudulent / Scam -> 6,505
NaN / unknown labels -> 13,152


## 11. Prepare the final modelling export

The following metadata columns are dropped for the modelling-ready file,
matching the original 3-class integration notebook's scope:

- `source_url`
- `source_platform`
- `data_source`


In [14]:
required_columns = [
    'title', 'company', 'location', 'industry_category', 'work_type',
    'salary_range', 'experience_level', 'min_qualification',
    'description', 'requirements', 'is_fraud'
]

model_df = final_df[required_columns].copy()

print('Final columns:')
print(model_df.columns.tolist())

for col in ['source_url', 'source_platform', 'data_source']:
    assert col not in model_df.columns, f'{col} was not removed.'
print('\nMetadata-column check passed.')


Final columns:
['title', 'company', 'location', 'industry_category', 'work_type', 'salary_range', 'experience_level', 'min_qualification', 'description', 'requirements', 'is_fraud']

Metadata-column check passed.


## 12. Missing-value overview (modelling columns)

In [15]:
missing_summary = (
    model_df.isna()
    .mean()
    .sort_values(ascending=False)
    .mul(100)
    .round(2)
    .to_frame('missing_percent')
)
display(missing_summary)


,missing_percent
salary_range,31.60
min_qualification,27.58
experience_level,26.10
requirements,22.46
is_fraud,18.71
work_type,14.39
company,12.62
industry_category,8.69
location,6.46
description,1.84


## 13. Preview the final dataset

In [16]:
display(model_df.head())
display(model_df.sample(min(10, len(model_df)), random_state=42))


,title,company,location,industry_category,work_type,salary_range,experience_level,min_qualification,description,requirements,is_fraud
0,Marketing Intern,"We're Food52, and we've created a groundbreaking and award-winning cooking site. We support, con...","US, NY, New York",NaN,Other,NaN,Internship,NaN,"Food52, a fast-growing, James Beard Award-winning online food community and crowd-sourced and cu...",Experience with content management systems a major plus (any blogging counts!)Familiar with the ...,0.0
1,Customer Service - Cloud Video Production,"90 Seconds, the worlds Cloud Video Production Service.90 Seconds is the worlds Cloud Video Produ...","NZ, , Auckland",Marketing and Advertising,Full-time,NaN,Not Applicable,NaN,Organised - Focused - Vibrant - Awesome!Do you have a passion for customer service? Slick typing...,"What we expect from you:Your key responsibility will be to communicate with the client, 90 Secon...",0.0
2,Commissioning Machinery Assistant (CMA),Valor Services provides Workforce Solutions that meet the needs of companies across the Private ...,"US, IA, Wever",NaN,NaN,NaN,NaN,NaN,"Our client, located in Houston, is actively seeking an experienced Commissioning Machinery Assis...",Implement pre-commissioning and commissioning procedures for rotary equipment.Execute all activi...,0.0
3,Account Executive - Washington DC,Our passion for improving quality of life through geography is at the heart of everything we do....,"US, DC, Washington",Computer Software,Full-time,NaN,Mid-Senior level,Bachelor's Degree,THE COMPANY: ESRI – Environmental Systems Research InstituteOur passion for improving quality of...,"EDUCATION: Bachelor’s or Master’s in GIS, business administration, or a related field, or equiva...",0.0
4,Bill Review Manager,SpotSource Solutions LLC is a Global Human Capital Management Consulting firm headquartered in M...,"US, FL, Fort Worth",Hospital & Health Care,Full-time,NaN,Mid-Senior level,Bachelor's Degree,"JOB TITLE: Itemization Review ManagerLOCATION: Fort Worth, TX ...","QUALIFICATIONS:RN license in the State of TexasDiploma or Bachelors of Science in Nursing, requi...",0.0


,title,company,location,industry_category,work_type,salary_range,experience_level,min_qualification,description,requirements,is_fraud
49778,Accounts Assistant,Swift HR Consultants Ltd,Thika,Retail,Full-time,"KES 95,000/month",No experience required,Diploma,Swift HR Consultants Ltd is looking for a Accounts Assistant to work in Thika. Maintain accurate...,Bachelor's degree required. 1-3 years preferred.,1.0
19312,"Programme Officer, BIOPAMA Action Component",NaN,PigiaMe,NaN,Full time,NaN,NaN,NaN,NaN,NaN,NaN
8053,Python Backend Developer,AGOGO creates a personalized audio channel by bringing together your favorite programming -- new...,"US, CA, San Francisco",Computer Software,Full-time,NaN,NaN,NaN,About AGOGOAGOGO is a personalized audio service that brings together your favorite programming ...,"The Ideal CandidateYou have a solid foundation in computer science, algorithms, and software des...",0.0
49928,Administrative Assistant,Jubilee Insurance Group,Kisii,Banking & Finance,Full-time,"KES 98,000/month",No experience required,Certificate,Jubilee Insurance Group is looking for a Administrative Assistant to work in Kisii. Coordinate w...,KCSE certificate required. 1-3 years preferred.,1.0
3186,Social Media Research Analyst,"BaaSSocial Media Marketing with IntelligenceΓια μια ολοκληρωμένη παρουσία στα Social Media, τα β...","GR, I, Athens",Marketing and Advertising,Full-time,NaN,Mid-Senior level,NaN,Key Responsibilities:Social Media Insights Analysis (Monitoring &amp; Reporting)Utilize Social M...,"Qualifications:Proficiency in social media monitoring, analytics and reporting, very high attent...",0.0
13661,PHP Developer,NaN,"CA, BC, Prince Rupert",Human Resources,Full-time,30-50,Mid-Senior level,Unspecified,seeking a strong php developer who is uber up on 5.2 and can handle the following:Let me give yo...,strong background in architecture and development with servers mixed in.,0.0
67001,Driver (Overseas),Vision HR Consultants Ltd,Oman,Hospitality,Full-time,USD 665/month,Entry level,Bachelor's degree,Vision HR Consultants Ltd is looking for a Driver (Overseas) to work in Oman. Assist in the dail...,Certificate required. No experience required preferred.,1.0
34015,Receptionist,Bamburi Cement,Mombasa,Banking & Finance,Full-time,"KES 37,000/month",Entry level,Bachelor's degree,Bamburi Cement is seeking a qualified Receptionist to join our team in Mombasa. Provide excellen...,Diploma required. No experience required preferred.,0.0
41327,Marketing Assistant,Jubilee Insurance,Nairobi,Healthcare,Contract,"KES 54,000/month",1-3 years,Certificate,Jubilee Insurance is seeking a qualified Marketing Assistant to join our team in Nairobi. Collab...,KCSE certificate required. 3-5 years preferred.,0.0
2299,QA Engineer,NaN,NaN,Information Services,Part-time,Oct-15,Associate,Bachelor's Degree,We are looking for a Quality Assurance Engineer to develop and execute exploratory tests as well...,"BS/MS degree in Computer Science, Engineering or a related subject3+ years of experience in soft...",0.0


## 14. Export
next stage : text preprocessing, feature engineering, train/test splitting, and multiclass fraud modelling.


In [17]:
output_file = 'merged_job_scam_3class.csv'
model_df.to_csv(output_file, index=False)
print(f'Saved: {output_file}')


Saved: merged_job_scam_3class.csv


## 15. Data Cleaning - synthetic_40k.csv

In [18]:
df1 = pd.read_csv('synthetic_40k.csv')
df1.shape

(40000, 13)

### 15.1 Initial inspection

In [19]:
print('Shape:', df1.shape)
print()
print(df1.dtypes)
print()
display(df1.head(10))

Shape: (40000, 13)

title                    str
company                  str
location                 str
industry_category        str
work_type                str
salary_range             str
experience_level         str
min_qualification        str
description              str
requirements             str
source_url           float64
source_platform          str
is_fraud               int64
dtype: object



,title,company,location,industry_category,work_type,salary_range,experience_level,min_qualification,description,requirements,source_url,source_platform,is_fraud
0,HR Assistant,Vision Manpower Kenya,Kisumu,Insurance,Full-time,"KES 339,000/month",3-5 years,Certificate,"Immediate start, no interview needed! HR Assistant needed in Kisumu. Send KES 3,000 via M-Pesa t...","No qualifications required, immediate hire.",NaN,Synthetic - AjiraCheck,2
1,Receptionist,Naivas Supermarket,Meru,Logistics,Full-time,"KES 35,000/month",Entry level,Certificate,Naivas Supermarket is seeking a qualified Receptionist to join our team in Meru. Collaborate wit...,Certificate required. Entry level preferred.,NaN,Synthetic - AjiraCheck,0
2,Cashier,Naivas Supermarket,Nyeri,Construction,Full-time,"KES 38,000/month",Entry level,Certificate,Naivas Supermarket is seeking a qualified Cashier to join our team in Nyeri. Handle customer inq...,Bachelor's degree required. No experience required preferred.,NaN,Synthetic - AjiraCheck,0
3,Security Guard,Vision HR Consultants Ltd,Qatar,Logistics,Full-time,USD 829/month,1-3 years,Bachelor's degree,Vision HR Consultants Ltd is looking for a Security Guard to work in Qatar. Ensure compliance wi...,Bachelor's degree required. Entry level preferred.,NaN,Synthetic - AjiraCheck,1
4,HR Assistant,Jubilee Insurance,Thika,Logistics,Full-time,"KES 83,000/month",3-5 years,Bachelor's degree,Jubilee Insurance is seeking a qualified HR Assistant to join our team in Thika. Support the tea...,Bachelor's degree required. Entry level preferred.,NaN,Synthetic - AjiraCheck,0
5,Warehouse Supervisor,KenGen,Nairobi,Insurance,Full-time,"KES 43,000/month",Entry level,Diploma,KenGen is seeking a qualified Warehouse Supervisor to join our team in Nairobi. Follow standard ...,KCSE certificate required. 3-5 years preferred.,NaN,Synthetic - AjiraCheck,0
6,Procurement Officer,Equity Bank Kenya,Meru,Retail,Full-time,"KES 48,000/month",1-3 years,Diploma,Equity Bank Kenya is seeking a qualified Procurement Officer to join our team in Meru. Collabora...,Bachelor's degree required. 1-3 years preferred.,NaN,Synthetic - AjiraCheck,0
7,Caregiver,Ready Manpower Limited,Saudi Arabia,Manufacturing,Full-time,USD 741/month,No experience required,Bachelor's degree,Ready Manpower Limited is seeking a qualified Caregiver to join our team in Saudi Arabia. Handle...,KCSE certificate required. Entry level preferred.,NaN,Synthetic - AjiraCheck,0
8,Domestic Worker,Top Level Management Ltd,United Arab Emirates,Telecommunications,Full-time,USD 428/month,3-5 years,KCSE certificate,Top Level Management Ltd is seeking a qualified Domestic Worker to join our team in United Arab ...,Certificate required. 1-3 years preferred.,NaN,Synthetic - AjiraCheck,0
9,Procurement Officer,Prime Manpower Services Ltd,Kisumu,Manufacturing,Contract,"KES 70,000/month",No experience required,KCSE certificate,Prime Manpower Services Ltd is looking for a Procurement Officer to work in Kisumu. Ensure timel...,Bachelor's degree required. 1-3 years preferred.,NaN,Synthetic - AjiraCheck,1


### 15.2 Standardize column names
Removing white space from the header row

In [20]:
df1.columns = df1.columns.str.strip()
print(df1.columns.tolist())

['title', 'company', 'location', 'industry_category', 'work_type', 'salary_range', 'experience_level', 'min_qualification', 'description', 'requirements', 'source_url', 'source_platform', 'is_fraud']


### 15.3 Trimming whitespace in text columns

In [21]:
text_cols_df1 = df1.select_dtypes(include='object').columns.tolist()
print('Text columns:', text_cols_df1)

for col in text_cols_df1:
    df1[col] = df1[col].astype('string').str.strip()

# Verify: count of remaining leading/trailing whitespace per column
ws_check = {col: (df1[col] != df1[col].str.strip()).sum() for col in text_cols_df1}
print('Remaining whitespace issues:', ws_check)

Text columns: ['title', 'company', 'location', 'industry_category', 'work_type', 'salary_range', 'experience_level', 'min_qualification', 'description', 'requirements', 'source_platform']


C:\Users\Administrator\AppData\Local\Temp\ipykernel_16652\2976624980.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  text_cols_df1 = df1.select_dtypes(include='object').columns.tolist()


Remaining whitespace issues: {'title': np.int64(0), 'company': np.int64(0), 'location': np.int64(0), 'industry_category': np.int64(0), 'work_type': np.int64(0), 'salary_range': np.int64(0), 'experience_level': np.int64(0), 'min_qualification': np.int64(0), 'description': np.int64(0), 'requirements': np.int64(0), 'source_platform': np.int64(0)}


### 15.4 Missing-value audit

"source_url" is 100% null across all 40,000 rows — it carries no information for the dataset, so we drop it rather than leaving it as a column of NaN.

In [22]:
missing_df1 = df1.isna().mean().mul(100).round(2).sort_values(ascending=False)
display(missing_df1.to_frame('missing_percent'))

fully_null_cols = missing_df1[missing_df1 == 100].index.tolist()
print('Fully-null columns to drop:', fully_null_cols)

df1 = df1.drop(columns=fully_null_cols)
print('Shape after dropping fully-null columns:', df1.shape)

,missing_percent
source_url,100.0
company,0.0
location,0.0
industry_category,0.0
title,0.0
work_type,0.0
salary_range,0.0
min_qualification,0.0
experience_level,0.0
description,0.0


Fully-null columns to drop: ['source_url']
Shape after dropping fully-null columns: (40000, 12)


### 15.5 Validate and cast "is_fraud"

Confirm the label only ever takes the values `0` (legitimate), `1` (suspicious), `2` (fraudulent) before casting to a small nullable integer type. Anything outside that set gets surfaced.

In [23]:
df1['is_fraud'] = pd.to_numeric(df1['is_fraud'], errors='coerce')

unexpected = df1.loc[~df1['is_fraud'].isin([0, 1, 2]), 'is_fraud']
print('Unexpected / non-numeric is_fraud values:', unexpected.unique())

df1['is_fraud'] = df1['is_fraud'].astype('Int8')
print(df1['is_fraud'].value_counts(dropna=False).sort_index())

Unexpected / non-numeric is_fraud values: []
is_fraud
0    22857
1    11429
2     5714
Name: count, dtype: Int64


### 15.6 Duplicate check

In [24]:
exact_dupes = df1.duplicated().sum()
print('Fully-identical duplicate rows:', exact_dupes)

content_key = ['title', 'company', 'location', 'description']
near_dupes = df1[df1.duplicated(subset=content_key, keep=False)]
print('Rows sharing title+company+location+description (salary differs):', len(near_dupes))

conflict_check = near_dupes.groupby(content_key)['is_fraud'].nunique()
print('Groups with conflicting is_fraud labels:', (conflict_check > 1).sum())

before = len(df1)
df1 = df1.drop_duplicates(keep='first').reset_index(drop=True)
print(f'Rows removed as exact duplicates: {before - len(df1)}')

Fully-identical duplicate rows: 0
Rows sharing title+company+location+description (salary differs): 530
Groups with conflicting is_fraud labels: 0
Rows removed as exact duplicates: 0


No fully-identical rows exist. There ARE 530 rows that share the same "title" / "company" / "location" / "description" but a different "salary_range"so these are kept rather than dropped. 

### 15.7 Parse "salary_range" into numeric fields

"salary_range" is a free-text string like "'KES 339,000/month'" or "'USD 829/month'". For modelling, split it into a "salary_currency", "salary_amount" (numeric), and "salary_period", then add a "salary_amount_kes" column normalized to Kenyan shillings so USD and KES postings are comparable.

**Assumption made explicit:** a fixed FX rate of 1 USD = 130 KES is used for the conversion. Adjust "USD_TO_KES" below if a different rate is preferred — it's isolated in one place on purpose.

In [25]:
import re

USD_TO_KES = 130  # <- adjust this single constant if a different FX rate is needed

salary_pattern = re.compile(r'^(KES|USD)\s*([\d,]+)\s*/\s*(\w+)$', flags=re.IGNORECASE)

def parse_salary(value):
    if pd.isna(value):
        return pd.Series([np.nan, np.nan, np.nan])
    m = salary_pattern.match(str(value).strip())
    if not m:
        return pd.Series([np.nan, np.nan, np.nan])
    currency, amount, period = m.groups()
    return pd.Series([currency.upper(), float(amount.replace(',', '')), period.lower()])

df1[['salary_currency', 'salary_amount', 'salary_period']] = df1['salary_range'].apply(parse_salary)

unparsed = df1['salary_range'].notna() & df1['salary_currency'].isna()
print('salary_range values that failed to parse:', unparsed.sum())
if unparsed.sum():
    display(df1.loc[unparsed, 'salary_range'].unique()[:10])

df1['salary_amount_kes'] = np.where(
    df1['salary_currency'] == 'USD',
    df1['salary_amount'] * USD_TO_KES,
    df1['salary_amount']
)

print(df1[['salary_range', 'salary_currency', 'salary_amount', 'salary_period', 'salary_amount_kes']].head())

salary_range values that failed to parse: 0
        salary_range salary_currency  salary_amount salary_period  \
0  KES 339,000/month             KES       339000.0         month   
1   KES 35,000/month             KES        35000.0         month   
2   KES 38,000/month             KES        38000.0         month   
3      USD 829/month             USD          829.0         month   
4   KES 83,000/month             KES        83000.0         month   

   salary_amount_kes  
0           339000.0  
1            35000.0  
2            38000.0  
3           107770.0  
4            83000.0  


### 15.8 Validate categorical fields

Full value set for each categorical column so any typo, stray category, or unexpected value is caught before modelling.

In [26]:
categorical_cols_df1 = ['work_type', 'experience_level', 'min_qualification', 'industry_category']

for col in categorical_cols_df1:
    print(f'--- {col} ({df1[col].nunique()} unique values) ---')
    print(df1[col].value_counts(dropna=False))
    print()

--- work_type (2 unique values) ---
work_type
Full-time    30028
Contract      9972
Name: count, dtype: Int64

--- experience_level (4 unique values) ---
experience_level
No experience required    10104
3-5 years                  9981
Entry level                9977
1-3 years                  9938
Name: count, dtype: Int64

--- min_qualification (4 unique values) ---
min_qualification
KCSE certificate     10140
Diploma              10024
Certificate           9960
Bachelor's degree     9876
Name: count, dtype: Int64

--- industry_category (9 unique values) ---
industry_category
Healthcare            4499
Manufacturing         4496
Retail                4475
Telecommunications    4465
Hospitality           4464
Banking & Finance     4439
Logistics             4425
Insurance             4371
Construction          4366
Name: count, dtype: Int64



### 15.9 Final check and export

In [37]:
print('Final shape:', df1.shape)
print()
print('Remaining missing values:')
display(df1.isna().sum().to_frame('missing_count'))

df1.to_csv('synthetic_40k_clean.csv', index=False)
print("\nSaved: synthetic_40k_clean.csv")
df1.duplicated().sum()

Final shape: (40000, 16)

Remaining missing values:


,missing_count
title,0
company,0
location,0
industry_category,0
work_type,0
salary_range,0
experience_level,0
min_qualification,0
description,0
requirements,0



Saved: synthetic_40k_clean.csv


np.int64(0)

## 16. Data cleaning — synthetic_augmentation_postings.csv (df2)

In [28]:
df2 = pd.read_csv('synthetic_augmentation_postings.csv')
print('Shape:', df2.shape)
print()
print(df2.dtypes)
print()
display(df2.head(3))

Shape: (5000, 14)

posting_id             str
title                  str
description            str
company                str
agency             float64
location               str
salary             float64
employment_type        str
email                  str
source             float64
country                str
is_overseas            str
is_fraud             int64
data_source            str
dtype: object



,posting_id,title,description,company,agency,location,salary,employment_type,email,source,country,is_overseas,is_fraud,data_source
0,SYN-000000,Customer Service Agent,Salary is commensurate with experience and will be discussed during the interview process. No fe...,"Williams, Nicholson and Davis",NaN,Machakos,NaN,Full-time,browningjason@hamilton.com,NaN,Kenya,No,0,synthetic_augmentation
1,SYN-000001,Data Entry Clerk,We are looking for a Data Entry Clerk to join our team based in Kakamega. The successful candida...,"Martinez, Walker and Burton",NaN,Kakamega,NaN,Part-time,rayrandy@williams.com,NaN,Kenya,No,0,synthetic_augmentation
2,SYN-000002,Sales Representative,Minimum requirements: a diploma or degree in a relevant field and at least 1 years of experience...,"Parker, Cabrera and White",NaN,Kakamega,NaN,Full-time,nathankelley@elliott.com,NaN,Kenya,No,0,synthetic_augmentation


### 16.1 Standardize column names and trim text fields

In [29]:
df2.columns = df2.columns.str.strip()

text_cols_df2 = df2.select_dtypes(include='object').columns.tolist()
print('Text columns:', text_cols_df2)

for col in text_cols_df2:
    df2[col] = df2[col].astype('string').str.strip()

Text columns: ['posting_id', 'title', 'description', 'company', 'location', 'employment_type', 'email', 'country', 'is_overseas', 'data_source']


C:\Users\Administrator\AppData\Local\Temp\ipykernel_16652\463795552.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  text_cols_df2 = df2.select_dtypes(include='object').columns.tolist()


### 16.2 Missing-value

In [30]:
missing_df2 = df2.isna().mean().mul(100).round(2).sort_values(ascending=False)
display(missing_df2.to_frame('missing_percent'))

fully_null_cols_df2 = missing_df2[missing_df2 == 100].index.tolist()
print('Fully-null columns to drop:', fully_null_cols_df2)
df2 = df2.drop(columns=fully_null_cols_df2)

df2['has_email'] = df2['email'].notna()
df2['email'] = df2['email'].fillna('not_provided')

print('\nShape after cleanup:', df2.shape)
print(df2['has_email'].value_counts())

,missing_percent
agency,100.00
salary,100.00
source,100.00
email,14.84
title,0.00
posting_id,0.00
location,0.00
company,0.00
description,0.00
employment_type,0.00


Fully-null columns to drop: ['agency', 'salary', 'source']

Shape after cleanup: (5000, 12)
has_email
True     4258
False     742
Name: count, dtype: int64


"agency", "salary", and "source" are 100% null across all 5,000 rows and so we drop them.

"email" is missing for ~15% of rows. Rather than dropping those rows, a "has_email" flag is kept and the NaNs are filled with an explicit placeholder instead of silently left blank.

### 16.3 Validate email format
Sanity-check every non-placeholder email against a basic "local@domain.tld" pattern and flag anything that doesn't match instead of assuming the column is clean.

In [31]:
email_pattern = re.compile(r'^[^@\s]+@[^@\s]+\.[^@\s]+$')

real_emails = df2.loc[df2['has_email'], 'email']
invalid_emails = real_emails[~real_emails.str.match(email_pattern)]
print(f'Invalid email formats: {len(invalid_emails)} out of {len(real_emails)}')
if len(invalid_emails):
    display(invalid_emails.head(10))

Invalid email formats: 0 out of 4258


### 16.4 Validate and cast 'is_fraud'

In [32]:
df2['is_fraud'] = pd.to_numeric(df2['is_fraud'], errors='coerce')

unexpected_df2 = df2.loc[~df2['is_fraud'].isin([0, 1, 2]), 'is_fraud']
print('Unexpected / non-numeric is_fraud values:', unexpected_df2.unique())

df2['is_fraud'] = df2['is_fraud'].astype('Int8')
print(df2['is_fraud'].value_counts(dropna=False).sort_index())

Unexpected / non-numeric is_fraud values: []
is_fraud
0    4258
1     535
2     207
Name: count, dtype: Int64


### 16.5 Check zero-variance columns

"country" ("'Kenya'") and "is_overseas" ("'No'") are constant across every row in this file. They're kept rather than dropped — they carry no information *within* this file alone, but they matter once this dataset is combined with sources that do vary on those fields — but it's worth flagging explicitly so it isn't mistaken for a bug later.

In [33]:
for col in ['country', 'is_overseas']:
    print(f'{col}: {df2[col].nunique()} unique value(s) -> {df2[col].unique()}')

country: 1 unique value(s) -> <StringArray>
['Kenya']
Length: 1, dtype: string
is_overseas: 1 unique value(s) -> <StringArray>
['No']
Length: 1, dtype: string


### 16.6 Remove duplicate postings

"posting_id" is unique for every row. Checking on "title" / "company" / "location" / "description" instead finds 10 pairs (20 rows) that are genuinely identical postings — same text, same employer, same location, same label — just issued a different ID. Those are true duplicates and are dropped, keeping the first occurrence.

In [ ]:
content_key_df2 = ['title', 'company', 'location', 'description']

dupe_mask = df2.duplicated(subset=content_key_df2, keep=False)
print('Rows involved in content duplicates:', dupe_mask.sum())
display(df2.loc[dupe_mask, ['posting_id'] + content_key_df2 + ['is_fraud']].sort_values(content_key_df2))
before_df2 = len(df2)
df2 = df2.drop_duplicates(subset=content_key_df2, keep='first').reset_index(drop=True)
print(f'\nRows removed as duplicate postings: {before_df2 - len(df2)}')

Rows involved in content duplicates: 20


,posting_id,title,company,location,description,is_fraud
2055,SYN-002055,Administrative Assistant,Not disclosed,Eldoret,We are looking for a Administrative Assistant to join our team based in Eldoret. The successful ...,1
4341,SYN-004341,Administrative Assistant,Not disclosed,Eldoret,We are looking for a Administrative Assistant to join our team based in Eldoret. The successful ...,1
2226,SYN-002226,Administrative Assistant,Not disclosed,Kakamega,We are looking for a Administrative Assistant to join our team based in Kakamega. The successful...,1
2257,SYN-002257,Administrative Assistant,Not disclosed,Kakamega,We are looking for a Administrative Assistant to join our team based in Kakamega. The successful...,1
4663,SYN-004663,Administrative Assistant,Not disclosed,Kisumu,Salary is commensurate with experience and will be discussed during the interview process. No fe...,1
4769,SYN-004769,Administrative Assistant,Not disclosed,Kisumu,Salary is commensurate with experience and will be discussed during the interview process. No fe...,1
1931,SYN-001931,Delivery Driver,Not disclosed,Kakamega,We are looking for a Delivery Driver to join our team based in Kakamega. The successful candidat...,1
4717,SYN-004717,Delivery Driver,Not disclosed,Kakamega,We are looking for a Delivery Driver to join our team based in Kakamega. The successful candidat...,1
948,SYN-000948,Delivery Driver,Not disclosed,Kisumu,We are looking for a Delivery Driver to join our team based in Kisumu. The successful candidate ...,1
2449,SYN-002449,Delivery Driver,Not disclosed,Kisumu,We are looking for a Delivery Driver to join our team based in Kisumu. The successful candidate ...,1



Rows removed as duplicate postings: 10


### 16.7 Final check and export

In [38]:
print('Final shape:', df2.shape)
print()
print('Remaining missing values:')
display(df2.isna().sum().to_frame('missing_count'))

df2.to_csv('synthetic_augmentation_postings_clean.csv', index=False)
print("\nSaved: synthetic_augmentation_postings_clean.csv")
df2.duplicated().sum()

Final shape: (4990, 12)

Remaining missing values:


,missing_count
posting_id,0
title,0
description,0
company,0
location,0
employment_type,0
email,0
country,0
is_overseas,0
is_fraud,0



Saved: synthetic_augmentation_postings_clean.csv


np.int64(0)